# Actividad 04 — Semana 03: Vistas, Optimización y SQL ↔ PySpark

**Semana:** 03  
**Tema:** CREATE VIEW, EXPLAIN y mapa SQL ↔ PySpark  
**Estudiante:** Daniel Guzmán  
**Notebook:** vistas_optimizacion_daniel  

## Objetivo

Cerrar la Semana 03 practicando tres conceptos clave:

- Creación de vistas con `CREATE VIEW`.
- Lectura de planes de ejecución con `EXPLAIN`.
- Equivalencias entre SQL y PySpark.

La tabla principal usada será:

- `workspace.silver.transactions_daniel`

También se usarán tablas Gold y Bronze sufijadas con `_daniel` para respetar la convención del entorno compartido.

In [0]:
USE CATALOG workspace;

SHOW TABLES IN silver;

In [0]:
SHOW TABLES IN gold;

In [0]:
SELECT
    COUNT(*) AS total_transacciones,
    COUNT(DISTINCT user_id) AS total_usuarios,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    SUM(CASE WHEN is_fraud IS NULL THEN 1 ELSE 0 END) AS sin_label
FROM silver.transactions_daniel;

In [0]:
CREATE OR REPLACE VIEW gold.v_analisis_fraude_daniel AS
SELECT
    transaction_id,
    transaction_date,
    mes,
    anio,
    hora,
    dia_semana,
    es_fin_de_semana,
    amount,
    amount_abs,
    merchant_id,
    merchant_category,
    mcc,
    card_type,
    is_fraud
FROM silver.transactions_daniel
WHERE is_fraud IS NOT NULL;

In [0]:
SELECT COUNT(*) AS total_registros
FROM gold.v_analisis_fraude_daniel;

In [0]:
DESCRIBE TABLE gold.v_analisis_fraude_daniel;

In [0]:
CREATE OR REPLACE VIEW gold.v_dashboard_fraude_daniel AS
WITH resumen AS (
    SELECT
        anio,
        mes,
        merchant_category,
        card_type,
        hora,
        es_fin_de_semana,
        COUNT(*) AS total_transacciones,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
        ROUND(
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) / COUNT(*) * 100,
            4
        ) AS tasa_fraude_pct,
        ROUND(SUM(amount_abs), 2) AS monto_total,
        ROUND(AVG(amount_abs), 2) AS ticket_promedio
    FROM silver.transactions_daniel
    WHERE is_fraud IS NOT NULL
    GROUP BY anio, mes, merchant_category, card_type, hora, es_fin_de_semana
)
SELECT *
FROM resumen;

In [0]:
SELECT *
FROM gold.v_dashboard_fraude_daniel
LIMIT 10;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW v_fraude_ultimos_90_dias AS
SELECT *
FROM silver.transactions_daniel
WHERE is_fraud = 1
  AND transaction_date >= DATE_ADD(
      (SELECT MAX(transaction_date) FROM silver.transactions_daniel),
      -90
  );

In [0]:
SELECT COUNT(*) AS fraudes_ultimos_90_dias
FROM v_fraude_ultimos_90_dias;

## Parte 1 — CREATE VIEW

Se crearon tres vistas:

- `gold.v_analisis_fraude_daniel`: vista persistente para el equipo de fraude, con columnas relevantes y solo transacciones etiquetadas.
- `gold.v_dashboard_fraude_daniel`: vista agregada para consumo analítico o dashboards.
- `v_fraude_ultimos_90_dias`: vista temporal para exploración dentro de la sesión actual.

### Diferencia entre CREATE VIEW y CREATE TEMPORARY VIEW

`CREATE VIEW` crea una vista persistente en el catálogo. Puede consultarse en otras sesiones mientras exista en el schema.

`CREATE TEMPORARY VIEW` crea una vista temporal disponible solo durante la sesión actual. Al cerrar la sesión o reiniciar el cluster, desaparece.

### ¿Cuándo usar una vista en lugar de una tabla Gold?

Usaría una vista cuando quiero exponer una consulta limpia sin duplicar datos físicamente, especialmente si la lógica es simple o si quiero evitar guardar otra tabla.

Usaría una tabla Gold cuando el cálculo es costoso, se consulta muchas veces o se necesita mejorar performance con datos pre-agregados.

### ¿Puede una vista referenciar a otra vista?

Sí. Una vista puede referenciar otra vista, siempre que la vista referenciada exista y el usuario tenga permisos para consultarla.

La vista temporal `v_fraude_ultimos_90_dias` encontró **424 fraudes** en los últimos 90 días del periodo registrado en el dataset.

In [0]:
EXPLAIN
SELECT merchant_category, COUNT(*) AS total
FROM silver.transactions_daniel
GROUP BY merchant_category
ORDER BY total DESC;

In [0]:
EXPLAIN EXTENDED
SELECT
    t.transaction_id,
    t.user_id,
    t.amount,
    t.is_fraud,
    u.birth_year
FROM silver.transactions_daniel t
INNER JOIN bronze.users_daniel u
    ON CAST(t.user_id AS INT) = CAST(u.id AS INT)
WHERE t.is_fraud = 1;

In [0]:
EXPLAIN FORMATTED
SELECT *
FROM silver.transactions_daniel
WHERE card_type = 'Credit'
  AND is_fraud = 1;

## Parte 2 — EXPLAIN

Se ejecutaron tres variantes de `EXPLAIN` para revisar el plan de ejecución de Spark SQL.

### EXPLAIN

Muestra una versión resumida del plan físico que Spark ejecutará. Es útil para identificar rápidamente operaciones como `Scan`, `Filter`, `HashAggregate`, `Exchange` y `Sort`.

### EXPLAIN EXTENDED

Muestra más detalle del proceso de optimización, incluyendo plan lógico, plan analizado, plan optimizado y plan físico. Es útil para entender cómo Catalyst transforma la consulta antes de ejecutarla.

### EXPLAIN FORMATTED

Presenta el plan físico en un formato más legible, separando nodos y detalles. Es útil para interpretar consultas complejas.

### BroadcastHashJoin

Un `BroadcastHashJoin` aparece cuando Spark decide copiar una tabla pequeña a todos los executors para evitar un shuffle costoso. Es útil al unir una tabla grande con una dimensión pequeña.

### SortMergeJoin

Un `SortMergeJoin` aparece normalmente cuando ambas tablas son grandes o no se puede hacer broadcast. Requiere ordenar y redistribuir datos por la clave de JOIN, por lo que suele implicar `Exchange` y ser más costoso que un broadcast join.

### Nodos principales

- `Scan`: lectura de datos.
- `Filter`: aplicación de filtros.
- `HashAggregate`: agregación.
- `Exchange`: movimiento de datos entre particiones, normalmente shuffle.
- `SortMergeJoin`: JOIN con ordenamiento y shuffle.
- `BroadcastHashJoin`: JOIN optimizado usando broadcast de una tabla pequeña.

In [0]:
%python
df = spark.table("workspace.silver.transactions_daniel")

display(
    df.select("transaction_id", "amount", "merchant_category", "is_fraud")
      .limit(5)
)

In [0]:
SELECT transaction_id, amount, merchant_category, is_fraud
FROM silver.transactions_daniel
LIMIT 5;

In [0]:
%python
from pyspark.sql import functions as F

conteo_fraudes_altos = (
    df.filter((F.col("is_fraud") == 1) & (F.col("amount") > 500))
      .count()
)

print(f"Fraudes con monto mayor a 500: {conteo_fraudes_altos}")

In [0]:
SELECT COUNT(*) AS fraudes_mayor_500
FROM silver.transactions_daniel
WHERE is_fraud = 1
  AND amount > 500;

In [0]:
%python
display(
    df.groupBy("merchant_category")
      .agg(
          F.count("*").alias("total"),
          F.round(F.avg("amount"), 2).alias("ticket_promedio"),
          F.sum("is_fraud").alias("fraudes")
      )
      .orderBy(F.col("total").desc())
      .limit(10)
)

In [0]:
SELECT
    merchant_category,
    COUNT(*) AS total,
    ROUND(AVG(amount), 2) AS ticket_promedio,
    SUM(is_fraud) AS fraudes
FROM silver.transactions_daniel
GROUP BY merchant_category
ORDER BY total DESC
LIMIT 10;

In [0]:
%python
from pyspark.sql import Window
from pyspark.sql import functions as F

window_spec = Window.partitionBy("card_type").orderBy(F.col("amount_abs").desc())

df_rank_pyspark = (
    df
    .withColumn("rank_gasto", F.rank().over(window_spec))
    .filter(F.col("rank_gasto") <= 3)
    .select("transaction_id", "card_type", "amount_abs", "merchant_category", "rank_gasto")
)

display(df_rank_pyspark)

In [0]:
SELECT
    transaction_id,
    card_type,
    amount_abs,
    merchant_category,
    RANK() OVER (
        PARTITION BY card_type
        ORDER BY amount_abs DESC
    ) AS rank_gasto
FROM silver.transactions_daniel
QUALIFY RANK() OVER (
    PARTITION BY card_type
    ORDER BY amount_abs DESC
) <= 3;

In [0]:
%python
MI_NOMBRE = "daniel"

resumen_pyspark = (
    df.groupBy("card_type")
      .agg(F.sum("is_fraud").alias("total_fraudes"))
)

resumen_pyspark.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.gold.resumen_por_tarjeta_pyspark_{MI_NOMBRE}")

display(resumen_pyspark)

In [0]:
CREATE OR REPLACE TABLE gold.resumen_por_tarjeta_sql_daniel AS
SELECT
    card_type,
    SUM(is_fraud) AS total_fraudes
FROM silver.transactions_daniel
GROUP BY card_type;

In [0]:
SHOW TABLES IN gold LIKE 'resumen_por_tarjeta*daniel';

In [0]:
SELECT
    es_fin_de_semana,
    COUNT(*) AS total_transacciones,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    ROUND(
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
        / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
        4
    ) AS tasa_fraude_pct
FROM silver.transactions_daniel
GROUP BY es_fin_de_semana
ORDER BY tasa_fraude_pct DESC;

In [0]:
%python
df_fin_semana = (
    df.groupBy("es_fin_de_semana")
      .agg(
          F.count("*").alias("total_transacciones"),
          F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
          F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas")
      )
      .withColumn(
          "tasa_fraude_pct",
          F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
      )
      .orderBy(F.col("tasa_fraude_pct").desc())
)

display(df_fin_semana)

## Parte 3 — Mapa SQL ↔ PySpark

Se implementaron equivalencias entre SQL y PySpark para operaciones comunes:

- Lectura y selección de columnas.
- Filtros.
- Agrupaciones.
- Window functions.
- Creación de tablas Delta.
- Ejemplo propio de Semana 02: fraude en fin de semana vs entre semana.

SQL resulta más legible para análisis directo, consultas agregadas y exploración por parte de analistas.

PySpark resulta más conveniente cuando la lógica hace parte de una pipeline, requiere varias transformaciones encadenadas o necesita control programático más detallado.

En el ejemplo propio, prefiero SQL para responder la pregunta de negocio porque la consulta es corta y fácil de leer. Preferiría PySpark si tuviera que integrar ese cálculo en una pipeline Medallón más grande.

## Parte 4 — Reflexión de cierre de semana

### ¿Cuándo usar SQL sobre Delta?

Usaría SQL sobre Delta para tareas analíticas, exploración de datos, validaciones rápidas, creación de vistas, consultas de negocio y agregaciones que puedan expresarse claramente con `SELECT`, `WHERE`, `GROUP BY`, `JOIN`, CTEs y window functions.

SQL es ideal cuando el consumidor principal es un analista o cuando la lógica se puede leer como una consulta directa sobre tablas.

### ¿Cuándo usar PySpark?

Usaría PySpark cuando necesito construir pipelines más completas, aplicar muchas transformaciones encadenadas, manejar lógica programática, reutilizar funciones, validar errores, escribir tablas Delta o integrar varios pasos dentro de un proceso de ingeniería de datos.

PySpark también es útil cuando la lógica requiere mayor control que una consulta SQL tradicional.

### Vistas vs tablas Gold

Una vista no duplica datos físicamente. Cada vez que se consulta, Spark ejecuta la query definida en la vista. Esto puede ser cómodo, pero si la vista tiene agregaciones pesadas puede ser más costosa en tiempo de ejecución.

Una tabla Gold pre-agregada sí guarda el resultado físicamente. Consultarla suele ser más rápido porque el cálculo ya fue materializado, pero ocupa almacenamiento y debe actualizarse cuando cambian los datos.

### Semana 04 — Datos actualizados cada hora

Si el equipo de fraude necesita datos actualizados cada hora, sí cambiaría la arquitectura.

Haría falta orquestar la pipeline para que se ejecute de forma periódica o incremental. También consideraría usar streaming o cargas incrementales para no reprocesar todo el dataset completo cada vez.

La arquitectura Medallón se mantiene, pero Bronze, Silver y Gold deberían actualizarse con una frecuencia definida y con controles de calidad automáticos.

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

df_tx = spark.table("workspace.silver.transactions_daniel")
df_users = spark.table("workspace.bronze.users_daniel")

print("DataFrames cargados")
print(f"Shuffle partitions configuradas: {spark.conf.get('spark.sql.shuffle.partitions')}")

In [0]:
%python
from pyspark.sql import functions as F

df_join_sin_broadcast_forzado = (
    df_tx.alias("t")
    .join(
        df_users.alias("u"),
        F.col("t.user_id").cast("int") == F.col("u.id").cast("int"),
        "left"
    )
)

print("=== JOIN sin hint de broadcast ===")
df_join_sin_broadcast_forzado.explain(mode="simple")

### Nota sobre configuración de broadcast

Se intentó modificar `spark.sql.autoBroadcastJoinThreshold` para comparar un JOIN con y sin broadcast automático, pero en este entorno de Databricks/Serverless esa configuración no está disponible.

Por eso se comparó:

- Un JOIN normal, sin hint explícito.
- Un JOIN con `broadcast()` forzado.

El objetivo se mantiene: observar cómo cambia el plan cuando Spark recibe una instrucción explícita para usar broadcast sobre una tabla pequeña.

In [0]:
%python
from pyspark.sql.functions import broadcast

df_join_con_broadcast = (
    df_tx.alias("t")
    .join(
        broadcast(df_users).alias("u"),
        F.col("t.user_id").cast("int") == F.col("u.id").cast("int"),
        "left"
    )
)

print("=== JOIN con broadcast() forzado ===")
df_join_con_broadcast.explain(mode="simple")

In [0]:
%python
print("=== Plan original ===")
df_tx.explain(mode="simple")

print("\n=== Plan después de repartition(32) ===")
df_repart = df_tx.repartition(32)
df_repart.explain(mode="simple")

print("\n=== Plan después de coalesce(4) ===")
df_coal = df_tx.coalesce(4)
df_coal.explain(mode="simple")

In [0]:
%python
distribucion = (
    df_tx
    .groupBy("merchant_city")
    .count()
    .orderBy(F.desc("count"))
)

display(distribucion.limit(20))

In [0]:
%python
top20_rows = distribucion.limit(20).collect()

top1 = top20_rows[0]["count"]
top20 = top20_rows[19]["count"]

print(f"Top 1 merchant_city: {top1:,} registros")
print(f"Top 20 merchant_city: {top20:,} registros")
print(f"Ratio top1/top20: {top1 / top20:.1f}x")

## Parte 5 — Shuffle, Broadcast Join y Skew

### Shuffle

Un shuffle ocurre cuando Spark necesita mover datos entre particiones o executors. Suele aparecer en operaciones como `groupBy`, `orderBy` y `JOIN`.

En los planes de ejecución se puede identificar buscando nodos como `Exchange`.

### BroadcastHashJoin vs SortMergeJoin

`BroadcastHashJoin` es conveniente cuando una tabla es pequeña, porque Spark la copia a los executors y evita redistribuir la tabla grande.

`SortMergeJoin` suele aparecer cuando ambas tablas son grandes o cuando el broadcast no está disponible. Puede ser más costoso porque requiere shuffle y ordenamiento por la clave de JOIN.

En esta actividad se comparó un JOIN sin broadcast automático contra un JOIN con `broadcast()` forzado usando `transactions_daniel` y `users_daniel`.

### Repartition vs coalesce

`repartition()` redistribuye los datos y puede aumentar o reducir particiones, pero implica shuffle.

`coalesce()` se usa principalmente para reducir particiones evitando un shuffle completo. Es útil antes de escribir archivos para no generar demasiados archivos pequeños.

### Skew

El skew ocurre cuando una clave concentra muchos más registros que las demás. Esto puede causar que una partición tarde mucho más que el resto.

Para detectarlo, se analizó la distribución de transacciones por `merchant_city` y se calculó el ratio entre el top 1 y el top 20. Si el ratio es muy alto, puede indicar riesgo de skew.

### Resultado de skew

En la distribución por `merchant_city`, el top 1 tiene **1,563,700 registros** y el top 20 tiene **49,562 registros**.

El ratio `top1/top20` fue de **31.6x**, lo que indica una distribución muy desigual y posible skew si se hicieran JOINs o agregaciones fuertes usando `merchant_city` como clave.